# ChadGPT — Train the MAOS model

Step-by-step LoRA fine-tune on one GH200. Run cells top to bottom.

- **Model:** pick Mistral-24B or Gemma-4-26B in the config cell (same notebook for both).
- **DiffusionGemma** trains differently (NeMo) — see the last section.
- The model learns the **MAOS behavior** (emotion-first, stops selling, handles objections) from `data/chat_train.jsonl`.

## 1. Install

In [ ]:
!pip install -q -U "torch==2.10.0" "torchvision==0.25.0" --index-url https://download.pytorch.org/whl/cu128
!pip install -q -U unsloth vllm trl datasets accelerate
import torch; print("cuda:", torch.cuda.is_available(), torch.__version__)

## 2. Config — set your model + knobs here

In [ ]:
MODEL   = "unsloth/Mistral-Small-24B-Instruct-2501"   # or "unsloth/gemma-4-26b-a4b-it"
MAXLEN  = 4096
BSZ     = 4          # gemma-4: use 2
GA      = 2          # gemma-4: use 8
LR      = 1.5e-4
EPOCHS  = 3          # a CEILING; early stopping picks the real number
TEACHER = "Qwen/Qwen2.5-32B-Instruct"
BONZO_DUMP = "data/raw/bonzo_new/conversations.jsonl"
OUT = "out/gemma_naf_lora" if "gemma" in MODEL.lower() else "out/mistral_naf_lora"
print("training", MODEL, "->", OUT)

## 3. Build the MAOS dataset

Produces `data/chat_train.jsonl` (+ val). Needs the teacher served. Skip if it already exists.

In [ ]:
import os
if not os.path.exists("data/chat_train.jsonl"):
    !TEACHER={TEACHER} BONZO_DUMP={BONZO_DUMP} bash nemo/build_dataset.sh
!wc -l data/chat_train.jsonl data/chat_val.jsonl

## 4. Load model + LoRA

In [ ]:
from unsloth import FastLanguageModel
TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
is_gemma = "gemma" in MODEL.lower()
if is_gemma:
    from unsloth import FastModel
    model, tok = FastModel.from_pretrained(MODEL, max_seq_length=MAXLEN, load_in_4bit=False, dtype=None)
    model = FastModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
        target_modules=TARGETS, finetune_vision_layers=False, finetune_language_layers=True,
        finetune_attention_modules=True, finetune_mlp_modules=True,
        use_gradient_checkpointing="unsloth", random_state=3407)
    INST, RESP = "<start_of_turn>user\n", "<start_of_turn>model\n"
else:
    model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=MAXLEN, load_in_4bit=False, dtype=None)
    model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
        target_modules=TARGETS, use_gradient_checkpointing="unsloth", random_state=3407)
    INST, RESP = "[INST]", "[/INST]"
print("loaded", MODEL)

## 5. Prepare data

In [ ]:
from datasets import load_dataset
def fmt(ex): return {"text": tok.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}
train_ds = load_dataset("json", data_files="data/chat_train.jsonl", split="train").map(fmt)
val_ds   = load_dataset("json", data_files="data/chat_val.jsonl",   split="train").map(fmt)
print(train_ds, val_ds); print(train_ds[0]["text"][:600])

## 6. Train

Evals every 100 steps, keeps the **best checkpoint by eval loss**, and **early-stops** — the eval curve picks the epoch count, not you.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only
cfg = SFTConfig(output_dir=OUT, per_device_train_batch_size=BSZ, gradient_accumulation_steps=GA,
    num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.03,
    logging_steps=10, bf16=True, max_length=MAXLEN, dataset_text_field="text", weight_decay=0.01,
    eval_strategy="steps", eval_steps=100, save_strategy="steps", save_steps=100, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False, report_to="none")
trainer = SFTTrainer(model=model, processing_class=tok, train_dataset=train_ds, eval_dataset=val_ds,
    args=cfg, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
trainer = train_on_responses_only(trainer, instruction_part=INST, response_part=RESP)
trainer.train()

## 7. Save + merge (for serving)

In [ ]:
model.save_pretrained(OUT); tok.save_pretrained(OUT)
model.save_pretrained_merged(OUT + "_merged", tok, save_method="merged_16bit")
print("adapter:", OUT, "| merged:", OUT + "_merged")

## 8. Quick sanity test
Generate on a grief situation right here — it must NOT ask a qualifying question.

In [ ]:
SYS = open("prompts/css_maos_system_prompt.txt").read()
ctx = (
  "--- CURRENT CONTEXT ---\n"
  "Stage: qualifying\nNext question key: property_use\nAnswers collected: N/A\n"
  "Ineligibility reason: N/A\nAgent name: Sarah\nClient name: Mike\n"
  "Client's latest message: I just lost my wife\nIs first message: no\n---\n\n"
  "SITUATION: the client shared something emotionally heavy. Do NOT ask a qualifying question, do NOT sell. Respond with brief genuine empathy."
)
msgs = [{"role":"system","content":SYS},{"role":"user","content":ctx}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(ids, max_new_tokens=300, do_sample=False)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 9. Full eval (optional — needs the model served)
```bash
vllm serve out/mistral_naf_lora_merged --served-model-name naf --dtype bfloat16 --max-model-len 8192 --port 8000
python3 eval/run_eval.py --base http://localhost:8000/v1 --model naf
```

## DiffusionGemma (separate — NeMo)
Block-diffusion model, trains via NeMo, same `data/chat_train.jsonl`:
```bash
pip install -U "transformers>=5.11.0" nemo-automodel torchdata
git clone https://github.com/NVIDIA-NeMo/Automodel.git
torchrun --standalone --nproc-per-node=1 Automodel/examples/dllm_sft/finetune.py -c nemo/diffusion_gemma_lora_1gpu.yaml
```
Train DiffusionGemma first, test it, then set `MODEL` above to Gemma-4 and re-run this notebook.